# Tokenización, Embeddings y Atención
## La mecánica interna de los Transformers

**Objetivo de aprendizaje:**  
Comprender cómo un modelo de lenguaje transforma texto en números,
qué representa cada paso de esa transformación, y por qué estas
decisiones de diseño determinan cuándo un LLM funciona bien y cuándo falla.

**Cadena completa:**
```
texto → tokens → embeddings → Q/K/V (atención) → predicción
```

---

In [1]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
import plotly.express as px
import plotly.graph_objects as go
import warnings, os
warnings.filterwarnings('ignore')

np.random.seed(42)
IMG = 'images'
os.makedirs(IMG, exist_ok=True)

print('numpy:', np.__version__)
print('pandas:', pd.__version__)
print('Setup OK')

numpy: 1.26.4
pandas: 2.2.3
Setup OK


---
## 1. Tokenización: el modelo no lee palabras, lee enteros

Un tokenizador divide el texto en fragmentos llamados **tokens** y asigna
a cada uno un identificador entero. Un token no es necesariamente una palabra:
puede ser una sílaba, un morfema, un signo de puntuación, o incluso un espacio.

El algoritmo más común es **BPE (Byte Pair Encoding)**: aprende qué fragmentos
son frecuentes en el corpus de entrenamiento y los trata como unidades.

In [2]:
# Vocabulario toy - ilustra el concepto, no es un tokenizador real
vocab = {
    'la': 543, 'el': 78, 'del': 79, 'de': 80, 'en': 81,
    'config': 8921, 'uración': 1204, 'sistema': 3301,
    'Spring': 4400, 'ter': 4401,
    'Kub': 9900, 'ernet': 9901, 'es': 82,
    '20': 2000, '24': 2001, '4': 2002, '.': 50,
    '587': 3000, '293': 3001, ',': 51, '12': 2003,
    'gato': 1100, 'perro': 1101, 'rey': 1200, 'reina': 1201,
    'hombre': 1300, 'mujer': 1301,
    'incumpl': 7700, 'imientos': 7701,
}

ejemplos = [
    ('"la configuración del sistema"',
     ['la', 'config', 'uración', 'del', 'sistema'],
     'texto normal - 5 tokens para 4 palabras'),
    ('"la empresa"',
     ['Spring', 'ter'],
     'nombre propio - 2 tokens, ninguno completo'),
    ('"Kubernetes"',
     ['Kub', 'ernet', 'es'],
     'término técnico - 3 tokens fragmentados'),
    ('"2024"',
     ['20', '24'],
     'número de 4 dígitos - 2 tokens sin valor numérico'),
    ('"4.587.293,12"',
     ['4', '.', '587', '.', '293', ',', '12'],
     'número con separadores - 7 tokens'),
    ('"incumplimientos"',
     ['incumpl', 'imientos'],
     'morfología española compleja - 2 tokens'),
]

print(f"{'Texto':<32} {'Tokens':<42} {'IDs':<28} Nota")
print('-' * 120)
for texto, tokens, nota in ejemplos:
    ids = str([vocab.get(t, '?') for t in tokens])
    print(f"{texto:<32} {str(tokens):<42} {ids:<28} {nota}")

Texto                            Tokens                                     IDs                          Nota
------------------------------------------------------------------------------------------------------------------------
"la configuración del sistema"   ['la', 'config', 'uración', 'del', 'sistema'] [543, 8921, 1204, 79, 3301]  texto normal -  5 tokens para 4 palabras
"la empresa"                     ['Spring', 'ter']                          [4400, 4401]                 nombre propio -  2 tokens, ninguno completo
"Kubernetes"                     ['Kub', 'ernet', 'es']                     [9900, 9901, 82]             término técnico -  3 tokens fragmentados
"2024"                           ['20', '24']                               [2000, 2001]                 número de 4 dígitos -  2 tokens sin valor numérico
"4.587.293,12"                   ['4', '.', '587', '.', '293', ',', '12']   [2002, 50, 3000, 50, 3001, 51, 2003] número con separadores -  7 tokens
"incumplimientos"    

In [3]:
# B05B_fig01: tabla tokenización asimétrica
fig, ax = plt.subplots(figsize=(13, 5))
ax.axis('off')

headers = ['Texto de entrada', 'Tokens resultantes', 'N tokens', 'N palabras', 'Ratio tok/pal']
rows = [
    ['"la configuración del sistema"', "['la','config','uración','del','sistema']",
     '5', '4', '1.25x'],
    ['"la empresa"', "['Spring','ter']", '2', '1', '2.0x'],
    ['"Kubernetes"', "['Kub','ernet','es']", '3', '1', '3.0x'],
    ['"2024"', "['20','24']", '2', '1 (núm.)', '2.0x'],
    ['"4.587.293,12"', "['4','.','587','.','293',',','12']", '7', '1 (núm.)', '7.0x'],
    ['"incumplimientos"', "['incumpl','imientos']", '2', '1', '2.0x'],
]
row_colors = [
    ['#ecf0f1']*5, ['#fdebd0']*5, ['#fdebd0']*5,
    ['#fadbd8']*5, ['#fadbd8']*5, ['#fdebd0']*5,
]
table = ax.table(
    cellText=rows, colLabels=headers,
    cellColours=row_colors, cellLoc='center', loc='center'
)
table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1, 1.7)
for (r, c), cell in table.get_celld().items():
    if r == 0:
        cell.set_facecolor('#2c3e50')
        cell.set_text_props(color='white', fontweight='bold')
ax.set_title(
    'Tokenización asimétrica: ratio tokens/palabra según tipo de texto',
    fontsize=12, fontweight='bold', pad=15
)
plt.tight_layout()
plt.savefig(f'{IMG}/B05B_fig01.png', dpi=150, bbox_inches='tight')
plt.close()
print('B05B_fig01.png guardada')

B05B_fig01.png guardada


In [4]:
# B05B_fig02: comparativa carácter vs token vs palabra
fig, ax = plt.subplots(figsize=(12, 3.5))
ax.axis('off')

headers2 = ['Unidad', 'Vocabulario', 'Robustez a palabras nuevas',
            'Eficiencia de contexto', 'Modelos representativos']
rows2 = [
    ['Carácter', 'Pequeño (~256)', 'Alta (nada es desconocido)',
     'Baja (secuencias muy largas)', 'Modelos tempranos, algunos multilingues'],
    ['Token BPE', 'Medio (32K-128K)', 'Media (subtokens siempre posibles)',
     'Alta (unidades óptimas del corpus)', 'GPT, Claude, Llama, BERT'],
    ['Palabra', 'Grande (>100K)', 'Baja (OOV frecuente)',
     'Muy alta', 'Word2Vec, GloVe (modelos clásicos)'],
]
rc2 = [['#d5e8d4']*5, ['#dae8fc']*5, ['#fff2cc']*5]
table2 = ax.table(
    cellText=rows2, colLabels=headers2,
    cellColours=rc2, cellLoc='center', loc='center'
)
table2.auto_set_font_size(False)
table2.set_fontsize(8.5)
table2.scale(1, 1.8)
for (r, c), cell in table2.get_celld().items():
    if r == 0:
        cell.set_facecolor('#2c3e50')
        cell.set_text_props(color='white', fontweight='bold')
ax.set_title(
    'Comparativa: carácter vs token (BPE) vs palabra como unidad mínima',
    fontsize=11, fontweight='bold', pad=10
)
plt.tight_layout()
plt.savefig(f'{IMG}/B05B_fig02.png', dpi=150, bbox_inches='tight')
plt.close()
print('B05B_fig02.png guardada')

B05B_fig02.png guardada


> **Antes de seguir:** Un cliente me dice que el modelo "no entiende bien
> los números en los presupuestos". ¿Cuál sería mi primer diagnóstico
> técnico antes de recomendar cambiar el modelo?

<details>
<summary>Orientación para el instructor (desplegar tras la reflexión)</summary>

Una respuesta madura menciona: la tokenización de números como fragmentos
sin valor numérico intrínseco, que el modelo no opera sobre valores sino
sobre representaciones de dígitos aislados, y que la aritmética no es
una capacidad estructural del mecanismo de atención.

Si nadie responde, preguntar: ¿cómo se representaría '4.587' en tokens?
¿Qué significa que cada fragmento tenga su propio embedding sin relación
numérica con los demás?

Señal de comprensión: el alumno distingue entre error de modelo y
limitación estructural de la representación. No propone cambiar de modelo;
evalúa si el caso de uso es adecuado para un LLM.

</details>

---
## 2. Embeddings: el modelo lee coordenadas, no texto

Cada token se convierte en un vector de alta dimensión llamado **embedding**.
La posición de ese vector en el espacio tiene significado semántico:
tokens con significados relacionados están cerca.

Usaremos vectores sintéticos de 50 dimensiones para ilustrar el concepto.
Los embeddings reales tienen entre 768 y 4096+ dimensiones.

In [5]:
# Embeddings sintéticos con estructura semántica
# (no son embeddings reales - ilustran el concepto de geometría semántica)

np.random.seed(42)
dim = 50

# Dimensiones semánticas base
d_animal  = np.random.randn(dim) * 0.4
d_royal   = np.random.randn(dim) * 0.4
d_male    = np.random.randn(dim) * 0.4
d_tech    = np.random.randn(dim) * 0.4
d_infra   = np.random.randn(dim) * 0.4

def emb(base, noise=0.04):
    return base + np.random.randn(dim) * noise

embeddings = {
    'gato':      emb(d_animal + d_male * 0.2),
    'perro':     emb(d_animal + d_male * 0.2),
    'felino':    emb(d_animal * 1.1),
    'mascota':   emb(d_animal * 0.6),
    'rey':       emb(d_royal + d_male),
    'reina':     emb(d_royal - d_male),
    'hombre':    emb(d_male),
    'mujer':     emb(-d_male),
    'Kubernetes':emb(d_tech + d_infra),
    'Docker':    emb(d_tech + d_infra * 0.9),
    'API':       emb(d_tech),
    'servidor':  emb(d_infra),
    'la empresa': emb(d_tech * 0.5 + d_infra * 0.3),
}

def cosine_sim(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

# Aritmética vectorial: rey - hombre + mujer ≈ reina
prediccion = embeddings['rey'] - embeddings['hombre'] + embeddings['mujer']
excluir = {'rey', 'hombre', 'mujer'}
sims = {w: cosine_sim(prediccion, v)
        for w, v in embeddings.items() if w not in excluir}

print('Aritmética vectorial: embedding(rey) - embedding(hombre) + embedding(mujer)')
print('Similitud coseno con el resultado predicho:')
print('-' * 45)
for w, s in sorted(sims.items(), key=lambda x: -x[1]):
    bar = '#' * max(0, int(s * 30))
    print(f'  {w:<12}: {s:+.3f}  {bar}')

mejor = max(sims, key=sims.get)
print(f'\nPalabra más cercana al resultado: {mejor} ({sims[mejor]:.3f})')

Aritmética vectorial: embedding(rey) - embedding(hombre) + embedding(mujer)
Similitud coseno con el resultado predicho:
---------------------------------------------
  reina       : +0.990  #############################
  mascota     : +0.138  ####
  felino      : +0.127  ###
  gato        : -0.028  
  perro       : -0.033  
  API         : -0.104  
  Docker      : -0.224  
  servidor    : -0.230  
  Kubernetes  : -0.235  
  la empresa  : -0.244  

Palabra más cercana al resultado: reina (0.990)


In [6]:
# B05B_fig03: visualización PCA 2D del espacio de embeddings
words = list(embeddings.keys())
matrix = np.array([embeddings[w] for w in words])

pca = PCA(n_components=2)
coords = pca.fit_transform(matrix)

grupos = {
    'Animales':       ['gato', 'perro', 'felino', 'mascota'],
    'Realeza/Género': ['rey', 'reina', 'hombre', 'mujer'],
    'Tecnología':     ['Kubernetes', 'Docker', 'API', 'servidor', 'la empresa'],
}
colores = {'Animales': '#e74c3c', 'Realeza/Género': '#3498db', 'Tecnología': '#27ae60'}

fig, ax = plt.subplots(figsize=(10, 7))
for grupo, palabras in grupos.items():
    idxs = [words.index(w) for w in palabras if w in words]
    ax.scatter(coords[idxs, 0], coords[idxs, 1],
               c=colores[grupo], s=130, label=grupo, zorder=3, edgecolors='white', lw=0.5)
    for i in idxs:
        ax.annotate(words[i], (coords[i, 0], coords[i, 1]),
                    xytext=(6, 6), textcoords='offset points', fontsize=9)

# Flechas relación rey → reina y hombre → mujer
for par in [('rey', 'reina'), ('hombre', 'mujer')]:
    i0, i1 = words.index(par[0]), words.index(par[1])
    ax.annotate('',
                xy=coords[i1], xytext=coords[i0],
                arrowprops=dict(arrowstyle='->', color='#7f8c8d', lw=1.5,
                                linestyle='dashed'))
    mid = (coords[i0] + coords[i1]) / 2
    ax.text(mid[0], mid[1], f'{par[0]}→{par[1]}',
            fontsize=8, color='#7f8c8d', ha='center')

var1 = pca.explained_variance_ratio_[0] * 100
var2 = pca.explained_variance_ratio_[1] * 100
ax.set_title(
    f'Espacio de embeddings 2D (PCA - {var1:.1f}% + {var2:.1f}% varianza)\n'
    'Palabras similares aparecen agrupadas - relaciones análogas muestran flechas paralelas',
    fontsize=11, fontweight='bold'
)
ax.set_xlabel(f'Componente principal 1 ({var1:.1f}%)')
ax.set_ylabel(f'Componente principal 2 ({var2:.1f}%)')
ax.legend(loc='best', fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{IMG}/B05B_fig03.png', dpi=150, bbox_inches='tight')
plt.close()
print('B05B_fig03.png guardada')

B05B_fig03.png guardada


---
## 3. Mecanismo de atención: cada token pondera la relevancia de los demás

La matriz de atención muestra cuánto peso da cada token a cada otro.
El valor en la fila *i*, columna *j* indica cuánto influye el token *j*
en la representación del token *i*.

Veremos dos variantes:
- **Bidireccional (BERT):** cada token ve toda la secuencia
- **Causal (GPT/Claude):** cada token solo ve los anteriores

In [7]:
# B05B_fig04: heatmaps de atención bidireccional (BERT) vs causal (GPT)
tokens_frase = ['El', 'contrato', 'vence', 'el', 'lunes', '.']

# Atención bidireccional sintética
# 'vence' atiende mucho a 'contrato' y 'lunes' (relación semántica principal)
attn_bi = np.array([
    [0.55, 0.15, 0.10, 0.08, 0.07, 0.05],   # El
    [0.12, 0.48, 0.18, 0.06, 0.11, 0.05],   # contrato
    [0.04, 0.32, 0.18, 0.06, 0.35, 0.05],   # vence (máx en contrato y lunes)
    [0.10, 0.10, 0.10, 0.52, 0.13, 0.05],   # el
    [0.05, 0.14, 0.26, 0.09, 0.41, 0.05],   # lunes
    [0.09, 0.09, 0.14, 0.09, 0.14, 0.45],   # .
], dtype=float)

# Atención causal: máscara triangular inferior, renormalizar filas
attn_causal = np.tril(attn_bi)
row_sums = attn_causal.sum(axis=1, keepdims=True)
attn_causal = attn_causal / row_sums
mask_upper = np.triu(np.ones(attn_causal.shape, dtype=bool), k=1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(attn_bi, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=tokens_frase, yticklabels=tokens_frase,
            ax=axes[0], vmin=0, vmax=0.6, cbar_kws={'shrink': 0.8})
axes[0].set_title('Atención bidireccional (BERT)\nCada token ve toda la secuencia',
                  fontweight='bold')
axes[0].set_xlabel('Token origen (Key)')
axes[0].set_ylabel('Token destino (Query)')

sns.heatmap(attn_causal, annot=True, fmt='.2f', cmap='Oranges',
            xticklabels=tokens_frase, yticklabels=tokens_frase,
            mask=mask_upper, ax=axes[1], vmin=0, vmax=0.9,
            cbar_kws={'shrink': 0.8})
axes[1].set_title('Atención causal (GPT / Claude)\nCada token solo ve los anteriores',
                  fontweight='bold')
axes[1].set_xlabel('Token origen (Key)')
axes[1].set_ylabel('Token destino (Query)')

plt.suptitle('"El contrato vence el lunes." - matrices de atención',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{IMG}/B05B_fig04.png', dpi=150, bbox_inches='tight')
plt.close()
print('B05B_fig04.png guardada')

B05B_fig04.png guardada


In [8]:
# B05B_fig05: tabla comparativa BERT vs GPT
fig, ax = plt.subplots(figsize=(13, 4.5))
ax.axis('off')

headers3 = ['', 'BERT (bidireccional)', 'GPT / Claude (causal)']
rows3 = [
    ['Puede ver al generar token N',
     'Tokens 1..N-1 y N+1..fin (todos)',
     'Solo tokens 1..N-1'],
    ['Objetivo de entrenamiento',
     'Predecir tokens enmascarados (MLM)',
     'Predecir el siguiente token (LM)'],
    ['Generación de texto libre',
     'No (sin capacidad autoregresiva)',
     'Sí (token a token)'],
    ['Clasificación / extracción',
     'Excelente (contexto completo bidireccional)',
     'Posible pero subóptimo'],
    ['Búsqueda semántica (indexado)',
     'Ideal (encoder produce embedding del doc)',
     'No directamente sin capa adicional'],
    ['Modelos representativos',
     'BERT, RoBERTa, Sentence-BERT',
     'GPT-4, Claude, Llama, Mistral'],
    ['Caso de uso la empresa',
     'Búsqueda en docs, clasificación de tickets',
     'Chatbot, generación de reportes, resumen'],
]
rc3 = [['#f5f5f5', '#dae8fc', '#d5e8d4']] * len(rows3)
table3 = ax.table(
    cellText=rows3, colLabels=headers3,
    cellColours=rc3, cellLoc='center', loc='center'
)
table3.auto_set_font_size(False)
table3.set_fontsize(8.5)
table3.scale(1, 1.75)
for (r, c), cell in table3.get_celld().items():
    if r == 0:
        cell.set_facecolor('#2c3e50')
        cell.set_text_props(color='white', fontweight='bold')
    if r > 0 and c == 0:
        cell.set_text_props(fontweight='bold')
ax.set_title(
    'Atención causal vs. bidireccional - decisión de diseño y consecuencias operativas',
    fontsize=11, fontweight='bold', pad=15
)
plt.tight_layout()
plt.savefig(f'{IMG}/B05B_fig05.png', dpi=150, bbox_inches='tight')
plt.close()
print('B05B_fig05.png guardada')

B05B_fig05.png guardada


---
## 4. Ventana de contexto: límite físico del mecanismo de atención

El coste del mecanismo de atención crece con el **cuadrado** de la longitud
de la secuencia. No es una restricción arbitraria: es consecuencia directa
de que cada token calcula su similitud con todos los demás: **O(n²)**.

In [9]:
# B05B_fig06: coste cuadrático de la atención
n_tokens = [512, 1000, 2048, 4096, 8000, 16000, 32000, 128000]
ops = [n**2 for n in n_tokens]
labels_n = [f'{n:,}' for n in n_tokens]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Izquierda: escala lineal
bar_colors = ['#3498db']*5 + ['#e74c3c']*3
bars = axes[0].bar(range(len(n_tokens)),
                   [o / 1e9 for o in ops], color=bar_colors)
axes[0].set_xticks(range(len(n_tokens)))
axes[0].set_xticklabels(labels_n, rotation=45, ha='right', fontsize=8)
axes[0].set_ylabel('Operaciones de atención (miles de millones)')
axes[0].set_xlabel('Tokens en la secuencia')
axes[0].set_title('Coste O(n²) - escala lineal', fontweight='bold')
axes[0].axvline(x=4.5, color='red', linestyle='--', alpha=0.5)
y_max = max(o / 1e9 for o in ops)
axes[0].text(4.6, y_max * 0.65, 'límite\ntípico\n2024',
             color='red', fontsize=8)
for bar, o in zip(bars, ops):
    h = bar.get_height()
    if h > 0.05:
        axes[0].text(bar.get_x() + bar.get_width() / 2,
                     h + y_max * 0.01,
                     f'{h:.1f}B', ha='center', fontsize=7)

# Derecha: escala log
axes[1].plot(range(len(n_tokens)),
             [o / 1e6 for o in ops],
             'o-', color='#2c3e50', linewidth=2, markersize=8)
axes[1].set_yscale('log')
axes[1].set_xticks(range(len(n_tokens)))
axes[1].set_xticklabels(labels_n, rotation=45, ha='right', fontsize=8)
axes[1].set_ylabel('Operaciones (millones) - escala log')
axes[1].set_xlabel('Tokens en la secuencia')
axes[1].set_title('Coste O(n²) - escala logarítmica', fontweight='bold')
axes[1].grid(True, alpha=0.3)
for i, (n, o) in enumerate(zip(n_tokens, ops)):
    label = f'{n:,}t -> {o/1e6:.0f}M'
    axes[1].annotate(label, (i, o / 1e6),
                     textcoords='offset points',
                     xytext=(8, 0), fontsize=7, color='#2c3e50')

plt.suptitle('Ventana de contexto y coste cuadrático de la atención',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{IMG}/B05B_fig06.png', dpi=150, bbox_inches='tight')
plt.close()
print('B05B_fig06.png guardada')

B05B_fig06.png guardada


In [10]:
# B05B_fig07.html: scatter interactivo embeddings PCA
grupo_por_palabra = {}
for g, palabras in grupos.items():
    for p in palabras:
        grupo_por_palabra[p] = g

df_plot = pd.DataFrame({
    'PC1': coords[:, 0],
    'PC2': coords[:, 1],
    'palabra': words,
    'grupo': [grupo_por_palabra.get(w, 'Otro') for w in words]
})

fig_plotly = px.scatter(
    df_plot, x='PC1', y='PC2', text='palabra', color='grupo',
    title=(
        'Espacio de embeddings 2D (PCA) - interactivo<br>'
        '<sup>Pasa el cursor sobre cada punto</sup>'
    ),
    color_discrete_map={
        'Animales': '#e74c3c',
        'Realeza/Género': '#3498db',
        'Tecnología': '#27ae60'
    },
    width=800, height=550
)
fig_plotly.update_traces(
    textposition='top center', marker_size=10
)
fig_plotly.update_layout(font=dict(size=12))

html_path = f'{IMG}/B05B_fig07.html'
fig_plotly.write_html(html_path)
print(f'B05B_fig07.html guardada')

try:
    fig_plotly.write_image(f'{IMG}/B05B_fig07.png')
    print('B05B_fig07.png guardada')
except Exception as e:
    print(f'PNG no disponible (kaleido): {e}')

fig_plotly.show()

B05B_fig07.html guardada
PNG no disponible (kaleido): 
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido



---
## 5. Ejercicio técnico: similitud coseno entre embeddings

**Enunciado:** Usando el vocabulario de embeddings definido arriba, calcula:

1. La matriz de similitud coseno para las 5 palabras del grupo Tecnología
2. Identifica qué dos palabras tecnológicas tienen embeddings más cercanos
3. Verifica si el resultado `rey - hombre + mujer` se acerca más a `reina`
   que a cualquier otro término del vocabulario

**Pista:** similitud coseno entre `a` y `b`:  
`sim = dot(a, b) / (norm(a) * norm(b))`  
Rango: [-1, 1]. Valor 1.0 = idénticos, 0.0 = ortogonales.

In [11]:
tech_words = ['Kubernetes', 'Docker', 'API', 'servidor', 'la empresa']
tech_vecs = np.array([embeddings[w] for w in tech_words])

# TODO 1: Normaliza tech_vecs fila a fila
# norms = ...
# tech_norm = ...

# TODO 2: Calcula la matriz de similitud (producto matricial de tech_norm)
# sim_matrix = ...

# TODO 3: Encuentra el par más similar (excluye la diagonal)
# ...

# TODO 4: Verifica la aritmética vectorial
# pred = ...
# ...

In [12]:
# SOLUCIÓN
tech_words = ['Kubernetes', 'Docker', 'API', 'servidor', 'la empresa']
tech_vecs = np.array([embeddings[w] for w in tech_words])

# Paso 1: normalizar
norms = np.linalg.norm(tech_vecs, axis=1, keepdims=True)
tech_norm = tech_vecs / norms

# Paso 2: matriz de similitud
sim_matrix = tech_norm @ tech_norm.T

print('Matriz de similitud coseno - Grupo Tecnología')
print('-' * 65)
header_row = ' ' * 12 + ''.join(f'{w:>13}' for w in tech_words)
print(header_row)
for i, w1 in enumerate(tech_words):
    row_str = f'{w1:>12}' + ''.join(
        f'{sim_matrix[i, j]:>13.3f}' for j in range(len(tech_words))
    )
    print(row_str)

# Paso 3: par más similar (excluye diagonal)
sim_no_diag = sim_matrix.copy()
np.fill_diagonal(sim_no_diag, -1.0)
i_max, j_max = np.unravel_index(sim_no_diag.argmax(), sim_no_diag.shape)
print(f'\nPar más similar: {tech_words[i_max]} <-> {tech_words[j_max]}'
      f' (similitud: {sim_matrix[i_max, j_max]:.3f})')

# Paso 4: aritmética vectorial
pred = embeddings['rey'] - embeddings['hombre'] + embeddings['mujer']
excluir4 = {'rey', 'hombre', 'mujer'}
sims4 = {
    w: cosine_sim(pred, v)
    for w, v in embeddings.items() if w not in excluir4
}
top3 = sorted(sims4.items(), key=lambda x: -x[1])[:3]
print('\nrey - hombre + mujer -> top 3 más cercanos:')
for w, s in top3:
    print(f'  {w:<12}: {s:+.3f}')

Matriz de similitud coseno -  Grupo Tecnología
-----------------------------------------------------------------
               Kubernetes       Docker          API     servidor   la empresa
  Kubernetes        1.000        0.995        0.629        0.760        0.956
      Docker        0.995        1.000        0.652        0.740        0.960
         API        0.629        0.652        1.000       -0.010        0.789
    servidor        0.760        0.740       -0.010        1.000        0.575
  la empresa        0.956        0.960        0.789        0.575        1.000

Par más similar: Kubernetes <-> Docker (similitud: 0.995)

rey - hombre + mujer -> top 3 más cercanos:
  reina       : +0.990
  mascota     : +0.138
  felino      : +0.127


---
## 6. Ejercicio de decisión: ventana de contexto como criterio de selección

### Caso: la empresa - automatización del análisis de contratos

El equipo de operaciones quiere automatizar la revisión de tres tipos de documentos:

**Tipo A - Órdenes de servicio:** 1-2 páginas (~500 tokens). Volumen: 200/día.

**Tipo B - Contratos de mantenimiento anual:** 15-25 páginas (~8.000 tokens).
Volumen: ~50/mes. Necesitan extraer: fecha de vencimiento, SLA comprometido
y cláusulas de penalización.

**Tipo C - Pliegos de condiciones de licitaciones:** 80-150 páginas
(~50.000-120.000 tokens). Volumen: 3-4/trimestre.

---

**Pregunta 1:** Para cada tipo, ¿qué estrategia recomendarías?
¿Envío directo al LLM, chunking + extracción, RAG, o solución más simple?
¿Por qué?

**Pregunta 2:** El responsable de IT propone usar Claude con 200K tokens de
contexto para los tres tipos "porque técnicamente caben". ¿Qué le responderías?

**Pregunta 3:** ¿Hay algún tipo donde NO recomendarías un LLM y preferirías
una solución más simple? ¿Por qué?

*[Escribe tu respuesta aquí - razonamiento, no código]*

<!-- CRITERIOS DE EVALUACIÓN (solo instructor)

Tipo A (500 tokens):
 - Envío directo. Sin problema de contexto. Coste bajo.
 - El alto volumen (200/día) hace relevante optimizar el prompt.

Tipo B (8.000 tokens):
 - Cabe en modelos modernos (8K-32K contexto). Envío completo válido.
 - Alternativa: chunking por secciones si el coste por token es relevante.

Tipo C (50K-120K tokens):
 - Aunque cabe en 200K, procesar pliegos completos tiene coste elevado.
 - La respuesta correcta es RAG: indexar, recuperar fragmentos relevantes,
    enviar solo el fragmento al LLM.
 - Señal de comprensión: el alumno menciona coste como criterio, no solo
    viabilidad técnica.

Pregunta 2 (modelo 200K para todos):
 - Técnicamente posible pero no óptimo.
 - El coste cuadrático hace que 120K tokens sea mucho más caro que 8K.
 - 'Lost in the middle': el modelo puede perder foco en documentos muy largos.
 - Señal de alarma: el alumno acepta sin cuestionar el coste operativo.

Pregunta 3 (cuándo NO usar LLM):
 - Si el contrato tiene estructura fija y el dato está siempre en la misma
    posición: un parser con regex es más rápido, barato y auditable.
 - Si el equipo legal necesita garantía de exactitud en fechas y valores:
    combinar extracción + verificación, no confiar solo en el LLM.
-->

---
## Assets generados

| Archivo | Contenido |
| ------- | --------- |
| `images/B05B_fig01.png` | Tabla: tokenización asimétrica por tipo de texto |
| `images/B05B_fig02.png` | Tabla: carácter vs token (BPE) vs palabra |
| `images/B05B_fig03.png` | Visualización PCA 2D del espacio de embeddings |
| `images/B05B_fig04.png` | Heatmaps: atención bidireccional (BERT) vs causal (GPT) |
| `images/B05B_fig05.png` | Tabla: BERT vs GPT/Claude - decisión operativa |
| `images/B05B_fig06.png` | Coste cuadrático O(n²) de la atención |
| `images/B05B_fig07.html` | Scatter interactivo: espacio de embeddings (Plotly) |